# 01 — Data pipeline (end-to-end)

**Phase 1** — Food.com → aliases → clean tables → enrichment → SQLite → match-rate chart

## Prerequisites
Place in `data/raw/food_com/`:
- `RAW_recipes.csv`
- `RAW_interactions.csv`

Optional external data (auto-downloaded or fallback):
- USDA FoodKeeper, FDC Foundation Foods, Open Food Facts sample

See `docs/DATA_DOWNLOADS.md`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW = ROOT / "data" / "raw" / "food_com"
CLEAN = ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

RECIPES_PATH = RAW / "RAW_recipes.csv"
INTERACTIONS_PATH = RAW / "RAW_interactions.csv"

for p, name in [(RECIPES_PATH, "RAW_recipes.csv"), (INTERACTIONS_PATH, "RAW_interactions.csv")]:
    status = "OK" if p.exists() else "MISSING — download from Kaggle"
    print(f"{name}: {status}")

## Step 1.1 — Inspect raw Food.com data

Run the cells below once files are in place.

In [ ]:
import pandas as pd

if not RECIPES_PATH.exists():
    raise FileNotFoundError(f"Download Food.com dataset to {RAW}")

recipes = pd.read_csv(RECIPES_PATH)
interactions = pd.read_csv(INTERACTIONS_PATH)

print("Recipes:", recipes.shape)
print("Interactions:", interactions.shape)
display(recipes.head(2))
display(interactions.head(2))

## Step 1.2 — Build ingredient alias dictionary

Expands normalisation coverage from the Food.com corpus (~1,500 aliases).

In [ ]:
import subprocess
from src.normalize import load_aliases

subprocess.run([sys.executable, str(ROOT / "scripts" / "build_ingredient_aliases.py")], check=True)
alias_count = load_aliases(ROOT / "assets" / "ingredient_aliases.csv")
print(f"Loaded {alias_count} ingredient aliases")

## Step 1.3–1.4 — Food.com cleaning and 5-core filter

Outputs: `clean_recipes.csv`, `clean_interactions.csv`

In [ ]:
import json
from src.data_pipeline import run_foodcom_pipeline

foodcom_stats = run_foodcom_pipeline(RAW, CLEAN)
print(json.dumps(foodcom_stats, indent=2))

recipes_clean = pd.read_csv(CLEAN / "clean_recipes.csv")
interactions_clean = pd.read_csv(CLEAN / "clean_interactions.csv")
print("\nClean recipes:", recipes_clean.shape)
print("Clean interactions:", interactions_clean.shape)
recipes_clean.head(3)

## Step 1.5–1.10 — Enrichment (shelf life, FDC, OFF, SQLite)

Downloads external datasets if missing; uses fallbacks when USDA endpoints block automated access.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "scripts" / "download_datasets.py")], check=False)

from src.enrichment_pipeline import run_enrichment_pipeline

enrich_stats = run_enrichment_pipeline(ROOT)
print(json.dumps(enrich_stats, indent=2))

## Step 1.11 — Match-rate analysis

Produces before/after normalisation metrics and chart for the report appendix.

In [ ]:
from src.match_rate import run_match_rate_analysis

match_stats = run_match_rate_analysis(ROOT)
print(json.dumps(match_stats, indent=2))
print(f"\nChart saved to: {match_stats['chart_path']}")